## Understanding Chaining And Runnables

### Load env file

In [1]:
from pprint import pprint
from dotenv import load_dotenv
import os
load_dotenv('../.env')

#print(os.getenv('LANGSMITH_API_KEY'))

True

### Create LLM Object

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
   base_url="http://localhost:11434",
   model="qwen2.5:latest",
   temperature=0.5,
   max_tokens=250
)

llm2 = ChatOllama(
   base_url="http://localhost:11434",
   model="llama3.2:latest",
   temperature=0.5,
   max_tokens=250
)


In [3]:
llm

ChatOllama(model='qwen2.5:latest', temperature=0.5, base_url='http://localhost:11434')

## Understanding Chaining and Runnables

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
  ("system", "You are an expert {fruit} farmer"),
  ("human", "What is the best way to grow {fruit}?")
])
prompt_template

# Without Chaining
# prompt = prompt_template.invoke({"fruit": "pineapple"})

# content = llm.invoke(prompt).content

# print(content)

# Chaining mechanism
chain = prompt_template | llm
chain.invoke({"fruit": "pineapple"})

AIMessage(content="Growing pineapples can be a rewarding endeavor, but it requires attention to several key factors. Here’s a step-by-step guide on how to grow pineapples successfully:\n\n### 1. Choosing the Right Variety\nDifferent varieties of pineapples are suited for different climates and growing conditions. Common types include:\n- **Smooth Cayenne**: One of the most widely grown commercial varieties.\n- **Queen**: Known for its sweet flavor, it's also suitable for home gardens.\n\n### 2. Climate Considerations\nPineapples thrive in warm, tropical or subtropical climates with temperatures between 65°F to 85°F (18°C to 29°C). They require at least 6 hours of direct sunlight per day and consistent warmth throughout the year.\n\n### 3. Soil Preparation\n- **Soil Type**: Pineapples grow best in well-drained, loamy soil with a pH between 4.5 and 6.0.\n- **Amending the Soil**: Amend your soil with compost or organic matter to improve fertility and drainage. Avoid heavy clay soils as th

### String Parsing

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate([
  ("system", "You are an expert {fruit} farmer"),
  ("human", "What is the best way to grow {fruit}?")
])
chain = prompt_template | llm | StrOutputParser()
response = chain.invoke({"fruit": "pineapple"})
pprint(response)

('Growing pineapples can be a rewarding experience, as they require relatively '
 'simple care once established. Here’s a step-by-step guide on how to grow '
 'pineapples:\n'
 '\n'
 '### 1. Choosing the Right Variety\n'
 '- **Dwarf Varieties:** These are often easier for home gardeners and include '
 'varieties like "Taylor," "Smooth Cayenne," or "Hilo Queen."\n'
 '- **Full-Sized Varieties:** Suitable if you have more space, such as '
 '"Pineapple Guiness" or "Kwai."\n'
 '\n'
 '### 2. Planting\n'
 '- **Plant Source:** You can use a pineapple crown (the top part with leaves) '
 'or small suckers from the base of an existing plant.\n'
 '- **Soil Preparation:** Pineapples prefer well-draining soil rich in organic '
 'matter. Amend your soil with compost and ensure it has good drainage to '
 'avoid root rot.\n'
 '\n'
 '### 3. Planting Location\n'
 '- **Sunlight:** Pineapples require full sun, at least 6 hours of direct '
 'sunlight per day.\n'
 '- **Climate:** They thrive in warm climates,


### Chaining Multiple Chains

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate([
  ("system", "You are an expert {fruit} farmer"),
  ("human", "What is the best way to grow {fruit}?")
])

# Chain 1
detailedResponseChain = prompt_template | llm | StrOutputParser()

headingInfoTemplate = ChatPromptTemplate.from_template("""
    Analyze the response and get me just the heading from the {response}
    
    Response should be in bullet points
    """)

# Chain 2
chainWithHeading = {"response": detailedResponseChain} | headingInfoTemplate | llm | StrOutputParser()

response = chainWithHeading.invoke({"fruit": "avocado"})

pprint(response)

('- Choose the Right Variety\n'
 '- Select a Suitable Site\n'
 '- Planting\n'
 '- Soil Preparation\n'
 '- Watering\n'
 '- Pruning\n'
 '- Pest and Disease Management\n'
 '- Harvesting\n'
 '- Pollination')


### Running Chains in Parallel

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

localMachineTemplate = ChatPromptTemplate([
    ("system", "You are an LLM expert"),
    ("user", "What is the advantage of running AI Models in {env}")
])

# Chain 1
localMachineChain = localMachineTemplate | llm | StrOutputParser()

cloudMachineTemplate = ChatPromptTemplate.from_template("""
                                         What is the advantage of running LLM in {machine}
                                         """)

# Chain 2
cloudMachineChain = cloudMachineTemplate | llm2 | StrOutputParser()

parallelRunnable = RunnableParallel(chain1=localMachineChain, chain2=cloudMachineChain)

response = parallelRunnable.invoke({"env": "local machine", "machine": "cloud machine"})

pprint(response['chain1'])
print("\n\n")
pprint(response['chain2'])

('Running AI models on a local machine offers several advantages, including:\n'
 '\n'
 '1. **Data Privacy and Security**: Running models locally ensures that '
 "sensitive data remains within your organization's network boundaries or even "
 'on a single device, which can enhance data privacy and security.\n'
 '\n'
 '2. **Control Over Data**: You have complete control over the data used in '
 'training and inference processes. This is particularly important when '
 'dealing with proprietary or regulated data.\n'
 '\n'
 '3. **Reduced Latency**: Local processing reduces latency compared to '
 "cloud-based solutions, as there's no need for data to travel over a network "
 'to reach remote servers. This can be crucial for real-time applications like '
 'autonomous vehicles or live video analysis.\n'
 '\n'
 '4. **Customization and Flexibility**: You have more flexibility in '
 'customizing the hardware and software environment according to your specific '
 'needs without being constrained b

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

prompt_template = ChatPromptTemplate([
    ("system", "You are an LLM expert"),
    ("user", "What is the advantage of running AI Models in {env}")
])

# Chain 1
detailedResponseChain = prompt_template | llm | StrOutputParser()

headingInfoTemplate = ChatPromptTemplate.from_template("""
                                         Analyse the response and get me just the heading from the {response}
                                         
                                         Response should be in bullet points
                                         """)
def choose_llm(response):
    response_text = str(response)
    if len(response_text) < 300:
        return llm2
    return llm

llm_selector = RunnableLambda(choose_llm)


# Chain 2
chainWithHeading = {"response": detailedResponseChain } | headingInfoTemplate | choose_llm | StrOutputParser()

response = chainWithHeading.invoke({"env": "local machine"})

pprint(response)

('- Data Privacy and Security\n'
 '- Control Over Data\n'
 '- Latency Reduction\n'
 '- Cost Efficiency\n'
 '- Offline Capabilities\n'
 '- Customization and Flexibility\n'
 '- Resource Management\n'
 '- Sustainability')
